# Fine-Tuning (Step Beyond Freezing)

**Theme:**

*“First use the brain as-is. Then carefully adapt it.”*

## What is Fine-Tuning means (Intuition)

Before:

- Only classifier learned

- Backbone stayed exactly as ImageNet

**Now:**

- We allow some deeper layers to adjust

- But keep early layers fixed

Why?

- Early layers = edges, textures (universal)

- Deep layers = object-specific (need adaptation)

## Unfreeze Last Block Only

In ResNet18, layers are structured like:
```bash
conv1
layer1
layer2
layer3
layer4  ← deepest block
fc
```

We will:

- Freeze everything

- Unfreeze only `layer4` + `fc`

### Imports

In [1]:
import torch 
import torch.nn as nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms, models

torch.__version__

'2.8.0+cu129'

### Modify code

In [2]:
# Load transform 
transform = transforms.Compose([
    transforms.Resize((224,224)),     # ResNet expects 224x224 pixel images 
    transforms.ToTensor()
])

# Load datasets 
train_data = datasets.CIFAR10(root='../week2/data', train=True, download=True, transform=transform) 
test_data = datasets.CIFAR10(root='../week2/data', train=False, download=True, transform=transform) 

# Load in dataloader 
train_loader = DataLoader(train_data, batch_size=64, shuffle=True) 
test_loader = DataLoader(test_data, batch_size=64, shuffle=False) 

# GPU availability 
if torch.cuda.is_available():
    device = 'cuda:0' 
    print("GPU is available") 
else:
    device = 'cpu' 
    print("GPU is not available") 

# Load model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

c:\Users\skroh\Desktop\Gen AI\.torchenv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


GPU is available


### Modules in ResNet

In [3]:
model._modules

{'conv1': Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False),
 'bn1': BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
 'relu': ReLU(inplace=True),
 'maxpool': MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False),
 'layer1': Sequential(
   (0): BasicBlock(
     (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   )
   (1): BasicBlock(
     (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (conv2): Conv2d(6

### Freezing and Unfreezing

In [6]:
# Freezing all the layers
for param in model.parameters():
    param.requires_grad = False 


# Unfreezing the params of layer 4 
for param in model.layer4.parameters():
    param.requires_grad = True 

# Handling fc layer 
model.fc = nn.Linear(model.fc.in_features, 10)   # now fc layer is trainable by default

In [7]:
# Loss function and optimizer 
loss_fn = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) 

**Very IMPORTANT** <br> 

Why smaller LR?

Because:

- Pretrained weights already good

- Large LR may destroy useful knowledge

## Again - Train & Evaluate

And compare:

- Previous accuracy: ~81%

- New accuracy: ?

It should improve.

### Training

runtime: 12mins+

In [8]:
model = model.to(device) 
model.train()

for epoch in range(5):
    train_loss = 0.0 
    for images, labels in train_loader:
        images = images.to(device) 
        labels = labels.to(device)
        # forward pass 
        outputs = model(images) 
        # compute loss 
        loss = loss_fn(outputs,labels) 

        # zero gradient
        optimizer.zero_grad() 
        # backward pass
        loss.backward() 
        # weights update 
        optimizer.step() 

        train_loss += loss.item() 

    print(f"Epoch: {epoch+1} | Train Loss: {train_loss/len(train_loader) :.5f}") 

Epoch: 1 | Train Loss: 0.42207
Epoch: 2 | Train Loss: 0.14094
Epoch: 3 | Train Loss: 0.04611
Epoch: 4 | Train Loss: 0.02154
Epoch: 5 | Train Loss: 0.01548


### Evaluation

In [9]:
model.eval() 

with torch.no_grad():
    correct = 0 
    total = 0 
    test_loss = 0.0

    for images,labels in test_loader:
        images = images.to(device)
        labels = labels.to(device) 

        outputs = model(images) 
        loss = loss_fn(outputs,labels) 
        predictions = outputs.argmax(dim=1) 

        correct += (predictions==labels).sum().item() 
        total += labels.size(0) 
        test_loss += loss.item() 

    print(f"Test Loss: {test_loss/len(test_loader) :.5f} | Accuracy: {correct/total * 100 : .2f} %")

Test Loss: 0.35164 | Accuracy:  90.63 %
